# Live ALPSS box plot

Helpers live in `aimdl_examples.live`.

In [1]:
import asyncio
import sys
from pathlib import Path

import ipywidgets as W
import numpy as np
import requests
from bokeh.io import output_notebook, push_notebook, show
from dotenv import find_dotenv, load_dotenv
from IPython.display import display

load_dotenv(find_dotenv())
sys.path.insert(0, str(Path.cwd().parent / 'src'))
output_notebook()   # load BokehJS into the notebook once

from aimdl_examples import get_girder_client
from aimdl_examples.live import (
    METRICS, TIMEFRAMES, VELOCITY,
    Config, GirderWebSocketSource, Store,
    build_bokeh_figure, build_material_map, build_view, new_item_ids,
    seed_store, update_bokeh,
)

session = requests.Session()
gc = get_girder_client(session=session)
print('connected:', gc.urlBase)

Loading BokehJS ...

connected: https://data.htmdec.org/api/v1/


In [2]:
# poll_seconds is now the *fallback heartbeat* cadence (§7): with push primary, the socket's
# own slow sync is the only polling left, so this runs slow (90 s) rather than the old 30.
# cfg = Config(metric=VELOCITY, seed_igsns=12, window_size=10, poll_seconds=10) #seeds
cfg = Config(metric=VELOCITY, seed_igsns=12, window_size=10, poll_seconds=10, max_seed_items=0) # no seeds

# Replay mode: hold back the newest rows and feed them in, to exercise the live path
# without waiting on the lab. "New" is an arrival counter, not a clock, so this drives
# the exact same highlight code the live path runs.
SIMULATE = False
SIM_HOLDBACK = 70
SIM_ROWS_PER_TICK = 4

In [3]:
store = Store(jitter=cfg.jitter)
material_map = build_material_map(gc)
print(f'material map: {len(material_map)} base IGSNs')

scanned = seed_store(gc, store, cfg)
print(f'seeded {len(store.rows)} rows / {len(store.igsns(cfg.metric))} IGSNs '
      f'from {scanned} items')

pending_rows = []
if SIMULATE:
    # hold back the newest rows; un-see them so the replay re-admits them
    held = sorted(store.rows, key=lambda r: r['arrival_seq'])[-SIM_HOLDBACK:]
    keep = {id(r) for r in held}
    store.rows = [r for r in store.rows if id(r) not in keep]
    for row in held:
        store.seen.discard(row['itemId'])
    pending_rows = held
    print(f'holding back {len(pending_rows)} rows for replay')


def replay_tick():
    global pending_rows
    if not pending_rows:
        return 'replay exhausted'
    batch, pending_rows = pending_rows[:SIM_ROWS_PER_TICK], pending_rows[SIM_ROWS_PER_TICK:]
    return f'replayed +{store.add(batch)} ({len(pending_rows)} left)'

material map: 8 base IGSNs
  ingested +0 | 0 without any metric | 0 errors
seeded 0 rows / 0 IGSNs from 0 items


In [4]:
w_metric = W.Dropdown(options=list(METRICS), value=cfg.metric, description='Metric')
w_highlight = W.Checkbox(value=True, description='Highlight new')
w_tf = W.Dropdown(options=list(TIMEFRAMES), value='last 24h', description='Timeframe')
w_x = W.Dropdown(options=['igsn', 'material_type'], value='igsn', description='x-axis')
w_size = W.IntSlider(value=cfg.window_size, min=3, max=25, description='Boxes')
w_follow = W.Checkbox(value=True, description='Follow live')
w_poll = W.BoundedIntText(value=cfg.poll_seconds, min=5, max=600, description='Heartbeat (s)')
b_prev = W.Button(description='◀ Older')
b_next = W.Button(description='Newer ▶')
b_stop = W.Button(description='Stop', button_style='danger')
status = W.HTML()

# The Bokeh figure is built once; each tick refills its ColumnDataSources in place and
# push_notebook(handle=...) ships only the diff over the plot's comm -- no PNG re-encode
# or Image.value swap. Like the old Image (and unlike an Output widget), the comm lands
# from any context, so the socket's background thread can repaint with no cell executing.
window_offset = 0
plot_handle = None
plot, sources = build_bokeh_figure(cfg)


def redraw(message=''):
    global window_offset
    if w_follow.value:
        window_offset = 0

    view = build_view(store, material_map, cfg, metric=w_metric.value, xcol=w_x.value,
                      timeframe=w_tf.value, window_size=w_size.value,
                      window_offset=window_offset)
    new_ids = new_item_ids(view.base, cfg.highlight_last_k) if w_highlight.value else set()
    update_bokeh(plot, sources, view, cfg, xcol=w_x.value, metric=w_metric.value,
                 new_ids=new_ids, highlight=w_highlight.value)

    shown = set(view.rows['itemId']) if not view.rows.empty else set()
    plot.title.text = (f'{view.stats["n_all"]} plottable | {view.stats["n_cats"]} '
                       f'{w_x.value} in timeframe | showing {len(view.categories)}')

    notes = [message]
    outside = len(new_ids - shown)
    if outside:
        # Highlight keys on arrival; the window filters on experiment time. A result
        # ingested now can carry an old experiment_date. Say so; never silently drop.
        notes.append(f'<b style="color:#d62728">{outside} new outside window</b>')
    if view.stats['n_no_ts']:
        notes.append(f'{view.stats["n_no_ts"]} rows lack experiment_date')
    if window_offset:
        notes.append(f'scrolled back {window_offset}')
    status.value = ' &nbsp;|&nbsp; '.join(n for n in notes if n)

    if plot_handle is not None:
        push_notebook(handle=plot_handle)


def shift(delta):
    def handler(_):
        global window_offset
        w_follow.value = False
        view = build_view(store, material_map, cfg, metric=w_metric.value, xcol=w_x.value,
                          timeframe=w_tf.value, window_size=w_size.value, window_offset=0)
        ceiling = max(0, view.stats['n_cats'] - w_size.value)
        window_offset = int(np.clip(window_offset + delta, 0, ceiling))
        redraw()
    return handler


b_prev.on_click(shift(+1))
b_next.on_click(shift(-1))
for widget in (w_metric, w_highlight, w_tf, w_x, w_size, w_follow):
    widget.observe(lambda change: redraw(), names='value')

display(W.VBox([
    W.HBox([w_metric, w_highlight, w_tf, w_x]),
    W.HBox([w_size, b_prev, b_next, w_follow]),
    W.HBox([w_poll, b_stop]),
    status,
]))
plot_handle = show(plot, notebook_handle=True)
redraw('seeded')

In [5]:
# Push is primary (S5, §4-§8): GirderWebSocketSource subscribes to the Girder firehose and
# ingests live rows with zero fetch. Its own slow `sync` (every w_poll seconds) plus a
# backfill on every (re)connect are now the ONLY polling left -- the old fast REST poll loop
# is retired. It owns background threads, so re-running this cell must stop the old one first
# -- two sockets would double every live redraw.
try:
    source.stop()
except NameError:
    pass

source = None
if not SIMULATE:
    source = GirderWebSocketSource(gc, store, cfg, render=redraw,
                                   heartbeat_seconds=int(w_poll.value)).start()
    # w_poll is the fallback heartbeat cadence; the source reads it fresh each cycle.
    w_poll.observe(lambda change: setattr(source, 'heartbeat_seconds', int(change['new'])),
                   names='value')
    print(f'subscribed to {source.url} | push primary, sync heartbeat every {w_poll.value}s')


# SIMULATE has no socket, so a small async loop still drives the replay feed. The live path
# needs no loop here: the socket pushes rows and its heartbeat is the safety net.
running = False


async def run_replay():
    global running
    running = True
    while running and pending_rows:
        redraw(replay_tick())
        await asyncio.sleep(3)
    running = False
    redraw('replay done' if not pending_rows else 'stopped')


def stop_all(_):
    globals().__setitem__('running', False)
    if source is not None:
        source.stop()


b_stop.on_click(stop_all)

# Re-running must not leave a previous replay loop alive: two run_replay() tasks means two
# redraws per tick, racing over the same figure.
try:
    task.cancel()
except NameError:
    pass
task = None
if SIMULATE:
    task = asyncio.ensure_future(run_replay())
    print('replay loop started -- scroll up for the plot')
else:
    print('live: socket push trigger started -- scroll up for the plot')

subscribed to wss://girder.htmdec.org/notifications/public | push primary, sync heartbeat every 10s
live: socket push trigger started -- scroll up for the plot
